In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('.')))

from sqlalchemy import text
from db.database import engine
from etl.constants import EUROPEAN_COUNTRIES, DECOUPLING_START_YEAR, DECOUPLING_END_YEAR

In [2]:
query = """
    SELECT 
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.co2_per_capita,
        e.co2_per_gdp,
        e.consumption_co2,
        e.consumption_co2_per_capita,
        e.consumption_co2_per_gdp,
        e.trade_co2,
        e.trade_co2_share,
        e.gdp,
        e.population,
        e.coal_co2,
        e.gas_co2,
        e.oil_co2,
        e.temperature_change_from_co2
    FROM emissions e
    JOIN countries c ON c.id = e.country_id
    WHERE e.year BETWEEN :start AND :end
    ORDER BY c.iso_code, e.year
"""

with engine.connect() as conn:
    df = pd.read_sql(
        text(query),
        conn,
        params={
            'start': DECOUPLING_START_YEAR,
            'end': DECOUPLING_END_YEAR
        }
    )

df_eu = df[df['iso_code'].isin(EUROPEAN_COUNTRIES)].copy()

print(f"Global dataset: {len(df)} rows, {df['iso_code'].nunique()} countries")
print(f"European dataset: {len(df_eu)} rows, {df_eu['iso_code'].nunique()} countries")

2026-06-07 15:13:04,256 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-06-07 15:13:04,259 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-07 15:13:04,261 INFO sqlalchemy.engine.Engine select current_schema()
2026-06-07 15:13:04,261 INFO sqlalchemy.engine.Engine [raw sql] {}


2026-06-07 15:13:04,261 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-06-07 15:13:04,271 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-07 15:13:04,280 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-07 15:13:04,280 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-06-07 15:13:04,280 INFO sqlalchemy.engine.Engine [generated in 0.00406s] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x000001E87A91C550>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-06-07 15:13:04,28

In [3]:
def build_index(df: pd.DataFrame, base_year: int = DECOUPLING_START_YEAR) -> pd.DataFrame:
    """
    Normalize GDP and CO2 to base_year = 100.
    Allows direct comparison across countries of different sizes.
    """
    df = df.copy()
    
    results = []
    for _, group in df.groupby('iso_code'):
        base = group[group['year'] == base_year]
        if base.empty:
            continue
        
        base_gdp = base['gdp'].values[0]
        base_co2 = base['co2_total'].values[0]
        base_consumption = base['consumption_co2'].values[0]
        
        # Skip if base year values are missing
        if pd.isna(base_gdp) or pd.isna(base_co2):
            continue
            
        group = group.copy()
        group['gdp_index'] = group['gdp'] / base_gdp * 100
        group['co2_index'] = group['co2_total'] / base_co2 * 100
        
        if not pd.isna(base_consumption):
            group['consumption_co2_index'] = group['consumption_co2'] / base_consumption * 100
        else:
            group['consumption_co2_index'] = np.nan
            
        results.append(group)
    
    return pd.concat(results, ignore_index=True)

df_indexed = build_index(df)
df_eu_indexed = build_index(df_eu)

print(f"Indexed dataset: {len(df_eu_indexed)} rows")
print(f"Sample — Poland 1995, 2000, 2005, 2010, 2015, 2020:")
print(df_eu_indexed[df_eu_indexed['iso_code'] == 'SWE'][
    ['year', 'gdp_index', 'co2_index', 'consumption_co2_index']
].query('year in [1995, 2000, 2005, 2010, 2015, 2020]').to_string())

Indexed dataset: 1156 rows
Sample — Poland 1995, 2000, 2005, 2010, 2015, 2020:
      year   gdp_index   co2_index  consumption_co2_index
1093  1995  104.789961  103.605204              96.080022
1098  2000  126.315321   95.602348              75.187440
1103  2005  147.563172   93.677831              77.299432
1108  2010  166.437284   92.357664              74.278098
1113  2015  178.591457   75.430622              61.645374
1118  2020  190.183443   63.782503              53.597696


In [4]:
# Finding countries with growing GDP and falling territorial CO2

latest = df_eu_indexed[df_eu_indexed['year'] == 2022].copy()
latest = latest.dropna(subset=['gdp_index', 'co2_index'])

# Classify countries
def classify_decoupling(row):
    gdp_grew = row['gdp_index'] > 120       # GDP grew more than 20%
    co2_fell = row['co2_index'] < 95        # CO2 fell more than 5%
    co2_stable = row['co2_index'] < 110     # CO2 roughly stable
    
    if gdp_grew and co2_fell:
        return 'Strong decoupling'
    elif gdp_grew and co2_stable:
        return 'Weak decoupling'
    elif gdp_grew:
        return 'No decoupling'
    else:
        return 'Economy shrank'

latest['decoupling_type'] = latest.apply(classify_decoupling, axis=1)

# Visualize
fig = px.scatter(
    latest,
    x='gdp_index',
    y='co2_index',
    color='decoupling_type',
    text='iso_code',
    title='Territorial Decoupling - European Countries (1990 = 100, measured at 2022)',
    labels={
        'gdp_index': 'GDP Index (1990 = 100)',
        'co2_index': 'CO2 Index (1990 = 100)'
    },
    color_discrete_map={
        'Strong decoupling': 'green',
        'Weak decoupling':   'orange',
        'No decoupling':     'red',
        'Economy shrank':    'gray'
    },
    height=600
)

# Reference lines
fig.add_hline(y=100, line_dash='dash', line_color='gray', annotation_text='CO2 baseline (1990)')
fig.add_vline(x=100, line_dash='dash', line_color='gray', annotation_text='GDP baseline (1990)')

# Green quadrant annotation
fig.add_annotation(
    x=260, y=33,
    text="Green quadrant<br>GDP up, CO2 down",
    showarrow=False,
    font=dict(color='green', size=12)
)

fig.update_traces(textposition='top center')
fig.show()

print("\nDecoupling classification:")
print(latest.groupby('decoupling_type')['country'].apply(list))


Decoupling classification:
decoupling_type
Economy shrank                                      [Georgia, Ukraine]
No decoupling                                [Cyprus, Ireland, Norway]
Strong decoupling    [Albania, Belgium, Bulgaria, Belarus, Switzerl...
Weak decoupling                                       [Austria, Spain]
Name: country, dtype: object


In [5]:
# Pick top 8 decouplers + Poland for context
top_decouplers = latest[latest['decoupling_type'] == 'Strong decoupling']['iso_code'].tolist()

# Always include Poland for context
focus_countries = top_decouplers[:8] + ['POL']
focus_countries = list(set(focus_countries))  # deduplicate

df_focus = df_eu_indexed[df_eu_indexed['iso_code'].isin(focus_countries)]

fig = px.line(
    df_focus,
    x='year',
    y='co2_index',
    color='iso_code',
    title='CO2 Trajectory (1990 = 100) - Top Decouplers',
    labels={'co2_index': 'CO2 Index (1990 = 100)', 'year': 'Year'},
    height=500
)
fig.add_hline(y=100, line_dash='dash', line_color='gray')
fig.show()

# #Consumption CO2
# fig = px.line(
#     df_focus,
#     x='year',
#     y='consumption_co2_index',
#     color='iso_code',
#     title='Consumption CO2 Trajectory (1990 = 100) - Top Decouplers',
#     labels={'co2_index': 'CO2 Index (1990 = 100)', 'year': 'Year'},
#     height=500
# )
# fig.add_hline(y=100, line_dash='dash', line_color='gray')
# fig.show()

# Same for GDP
fig2 = px.line(
    df_focus,
    x='year',
    y='gdp_index',
    color='iso_code',
    title='GDP Trajectory (1990 = 100) - Top Decouplers',
    labels={'gdp_index': 'GDP Index (1990 = 100)', 'year': 'Year'},
    height=500
)
fig2.add_hline(y=100, line_dash='dash', line_color='gray')
fig2.show()

In [6]:
# Territorial CO2 vs consumption CO2
# Consumption stayed high = fake decoupling

latest_full = df_eu_indexed[
    (df_eu_indexed['year'] == 2022) &
    (df_eu_indexed['consumption_co2_index'].notna())
].copy()

latest_full['gap'] = latest_full['co2_index'] - latest_full['consumption_co2_index']
# Positive gap = territorial looks better than reality (imported emissions)
# Negative gap = territorial looks worse than reality (exported emissions)

latest_full = latest_full.sort_values('gap', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Territorial<br>CO2 index',
    x=latest_full['iso_code'],
    y=latest_full['co2_index'],
    marker_color='steelblue'
))

fig.add_trace(go.Bar(
    name='Consumption<br>CO2 index',
    x=latest_full['iso_code'],
    y=latest_full['consumption_co2_index'],
    marker_color='coral'
))

fig.add_hline(y=100, line_dash='dash', line_color='gray')
fig.update_layout(
    title='Territorial vs Consumption CO2 - 2022<br><sup>Gap = difference between what countries emit vs what they consume</sup>',
    barmode='group',
    height=600,
    xaxis_tickangle=-45
)
fig.show()

print("\nBiggest 'fake decouplers' (territorial looks better than reality):")
print(latest_full[latest_full['gap'] > 2][['country', 'co2_index', 'consumption_co2_index', 'gap']].to_string())

print("\nCountries that look worse than reality (exporters):")
print(latest_full[latest_full['gap'] < -30][['country', 'co2_index', 'consumption_co2_index', 'gap']].to_string())


Biggest 'fake decouplers' (territorial looks better than reality):
      country  co2_index  consumption_co2_index       gap
1120   Sweden  63.294843              59.142009  4.152834
440   Finland  63.837599              59.534690  4.302910
66    Austria  98.814941              93.802594  5.012347

Countries that look worse than reality (exporters):
         country  co2_index  consumption_co2_index         gap
848        Malta  73.094355             493.993258 -420.898903
100      Belgium  73.889558             172.101217  -98.211659
542      Georgia  80.808081             164.234857  -83.426776
202  Switzerland  74.635892             144.851712  -70.215820
168      Belarus  52.665984             104.387680  -51.721696
610      Croatia  76.840725             118.812226  -41.971501
814       Latvia  33.607975              68.117394  -34.509420
338      Denmark  54.404697              86.629132  -32.224435


In [ ]:
# 3 trajectories for key countries at the same time

def plot_three_trajectories(iso_codes: list, df_indexed: pd.DataFrame, title: str):
    """
    For each country show GDP, territorial CO2 and consumption CO2
    on the same chart (1990 = 100)
    """
    fig = make_subplots(
        rows=len(iso_codes) // 2 + len(iso_codes) % 2,
        cols=2,
        subplot_titles=[
            df_indexed[df_indexed['iso_code'] == iso]['country'].iloc[0]
            for iso in iso_codes
        ]
    )

    for i, iso in enumerate(iso_codes):
        row = i // 2 + 1
        col = i % 2 + 1

        country_df = df_indexed[
            df_indexed['iso_code'] == iso
        ].dropna(subset=['gdp_index'])

        # GDP
        fig.add_trace(go.Scatter(
            x=country_df['year'],
            y=country_df['gdp_index'],
            name='GDP',
            line=dict(color='blue', width=2),
            showlegend=(i == 0)
        ), row=row, col=col)

        # Territorial CO2
        fig.add_trace(go.Scatter(
            x=country_df['year'],
            y=country_df['co2_index'],
            name='Territorial CO2',
            line=dict(color='red', width=2),
            showlegend=(i == 0)
        ), row=row, col=col)

        # Consumption CO2
        fig.add_trace(go.Scatter(
            x=country_df['year'],
            y=country_df['consumption_co2_index'],
            name='Consumption CO2',
            line=dict(color='orange', width=2, dash='dash'),
            showlegend=(i == 0)
        ), row=row, col=col)

        # Baseline
        fig.add_hline(
            y=100,
            line_dash='dot',
            line_color='gray',
            row=row, col=col
        )

    fig.update_layout(
        height=310 * (len(iso_codes) // 2 + 1),
        title_text=title
    )
    fig.show()


# Biggest importers - do their consumption curves expose fake decoupling?
importers = ['GBR', 'DEU', 'FRA', 'ITA', 'CHE', 'BEL']
plot_three_trajectories(
    importers,
    df_eu_indexed,
    'Fake decoupling test - biggest emission importers<br>'
    '<sup>If consumption CO2 (orange) stays high while territorial (red) drops - fake decoupling</sup><br>'
)

In [59]:
# Best territorial performers — is their decoupling real?
best_performers = ['PRT', 'HUN', 'SWE', 'ROU', 'DNK', 'SVK']
plot_three_trajectories(
    best_performers,
    df_eu_indexed,
    'Real decoupling test - best territorial performers<br>'
    '<sup>If consumption CO2 (orange) also drops - decoupling is real</sup>'
)

In [10]:
# Interesting outliers
outliers = ['POL', 'IRL', 'CYP', 'GEO']
plot_three_trajectories(
    outliers,
    df_eu_indexed,
    'Outliers and interesting cases'
)

In [ ]:
#Special one for Norway
query = """
    SELECT 
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.co2_per_capita,
        e.co2_per_gdp,
        e.consumption_co2,
        e.consumption_co2_per_capita,
        e.consumption_co2_per_gdp,
        e.trade_co2,
        e.trade_co2_share,
        e.gdp,
        e.population
    FROM emissions e
    JOIN countries c ON c.id = e.country_id
    WHERE e.year BETWEEN :start AND :end AND c.iso_code = 'NOR'
    ORDER BY c.iso_code, e.year
"""

with engine.connect() as conn:
    df_nor = pd.read_sql(
        text(query),
        conn,
        params={
            'start': 2003,
            'end': DECOUPLING_END_YEAR
        }
    )

nor = build_index(df_nor, 2003).copy()

fig = go.Figure()

fig.add_trace(go.Scatter(x=nor['year'], y=nor['gdp_index'],
    name='GDP', line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=nor['year'], y=nor['co2_index'],
    name='Territorial CO2', line=dict(color='red', width=2)))
fig.add_trace(go.Scatter(x=nor['year'], y=nor['consumption_co2_index'],
    name='Consumption CO2', line=dict(color='orange', width=2, dash='dash')))
fig.add_hline(y=100, line_dash='dot', line_color='gray')

fig.update_layout(
    height=500,
    title_text='Decopling test for Norway (2003 = 100)<br>'
               '<sup>*cause for consumption co2 Norway has data only from 2003</sup>'
)
fig.show()


# Norway hydrocarbon production
nor = df_indexed[df_indexed['iso_code'] == 'NOR'].copy()
fig = go.Figure()
fig.add_trace(go.Scatter(x=nor['year'], y=nor['coal_co2'],
    name='Coal', line=dict(color='black')))
fig.add_trace(go.Scatter(x=nor['year'], y=nor['gas_co2'],
    name='Gas', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=nor['year'], y=nor['oil_co2'],
    name='Oil', line=dict(color='brown')))

fig.update_layout(
    height=500,
    title_text='Norway - The Oil Exporter Paradox<br>'
               '<sup>GDP from oil exports, but domestic energy is almost 100% hydro</sup>'
)
fig.show()

2026-06-07 15:37:44,741 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-07 15:37:44,745 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-06-07 15:37:44,751 INFO sqlalchemy.engine.Engine [cached since 1480s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x000001E801E12790>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-06-07 15:37:44,755 INFO sqlalchemy.engine.Engine 
    SELECT 
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_total,
        e.co2_per_capita

In [39]:
# trade_co2 positive = net importer of emissions
# trade_co2 negative = net exporter of emissions

df_trade = df_eu[
    (df_eu['year'] >= 1990) &
    (df_eu['trade_co2'].notna())
].copy()

# Latest year available
latest_trade = df_trade[df_trade['year'] == df_trade['year'].max()].copy()
latest_trade = latest_trade.sort_values('trade_co2', ascending=True)

colors = ['green' if x < 0 else 'red' for x in latest_trade['trade_co2']]

fig = go.Figure(go.Bar(
    x=latest_trade['iso_code'],
    y=latest_trade['trade_co2'],
    marker_color=colors,
    text=latest_trade['country'],
))

fig.add_hline(y=0, line_color='black')
fig.update_layout(
    title='Net Trade CO2 - European Countries<br><sup>Negative = exporting emissions | Positive = importing emissions</sup>',
    yaxis_title='Net CO2 from trade (million tonnes)',
    height=500,
    xaxis_tickangle=-45
)
fig.show()

# Trade CO2 as % of total
latest_trade['trade_impact'] = (latest_trade['trade_co2'] / latest_trade['co2_total'] * 100).round(1)

print("\nTrade CO2 as % of territorial emissions:")
print(latest_trade[['country', 'trade_co2', 'trade_co2_share', 'trade_impact']]
      .sort_values('trade_impact').to_string())


Trade CO2 as % of territorial emissions:
             country  trade_co2  trade_co2_share  trade_impact
5303          Poland     -9.388           -3.314          -3.3
4929          Norway      0.101            0.260           0.3
713         Bulgaria      0.809            2.345           2.3
3943      Luxembourg      0.491            7.181           7.2
849          Belarus      6.812           12.222          12.2
6153        Slovakia      4.269           13.875          13.9
5541         Romania     10.385           15.303          15.3
6901         Ukraine     26.980           19.365          19.4
1733          Cyprus      1.436           19.982          20.0
1767         Czechia     16.793           20.176          20.2
2107           Spain     45.155           20.957          21.0
3059         Ireland      7.266           21.645          21.6
169          Albania      0.965           21.838          21.8
5371        Portugal      9.838           26.149          26.1
4895     Neth

In [45]:
# Trade CO2 over time
top_importers = ['GBR', 'DEU', 'FRA', 'ITA', 'CHE']
top_exporters = ['POL']  # only real exporter of co2 in Europe
focus = top_importers + top_exporters

df_trade_time = df_eu[
    (df_eu['iso_code'].isin(focus)) &
    (df_eu['trade_co2'].notna())
].copy()

fig = px.line(
    df_trade_time,
    x='year',
    y='trade_co2',
    color='iso_code',
    title='Trade CO2 Over Time - Importers vs Poland<br>'
          '<sup>Positive = importing emissions | Negative = exporting</sup>',
    labels={'trade_co2': 'Net CO2 from trade (Mt)', 'year': 'Year'},
    height=500
)
fig.add_hline(y=0, line_dash='dash', line_color='black',
              annotation_text='Zero line')

# Annotate the key insight
fig.add_annotation(
    x=2007, y=-12,
    text="Poland - only one net<br>exporter in Europe",
    showarrow=True,
    arrowhead=2,
    font=dict(color='green')
)
fig.show()

In [51]:
# Germany: honest decoupler
deu = df_indexed[df_indexed['iso_code'] == 'DEU'].copy()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Three trajectories (1990 = 100)',
        'Absolute: Territorial vs Consumption CO2',
        'Energy mix - what drove the reduction?',
        'Trade CO2 - growing import dependency'
    ]
)

# Three trajectories
fig.add_trace(go.Scatter(x=deu['year'], y=deu['gdp_index'],
    name='GDP', line=dict(color='blue', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['co2_index'],
    name='Territorial CO2', line=dict(color='red', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['consumption_co2_index'],
    name='Consumption CO2', line=dict(color='orange', width=2, dash='dash')), row=1, col=1)
fig.add_hline(y=100, line_dash='dot', line_color='gray', row=1, col=1)

# Absolute values
fig.add_trace(go.Scatter(x=deu['year'], y=deu['co2_total'],
    name='Territorial (Mt)', line=dict(color='red'),
    showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['consumption_co2'],
    name='Consumption (Mt)', line=dict(color='orange', dash='dash'),
    showlegend=False), row=1, col=2)

# Energy mix
fig.add_trace(go.Scatter(x=deu['year'], y=deu['coal_co2'],
    name='Coal', line=dict(color='black')), row=2, col=1)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['gas_co2'],
    name='Gas', line=dict(color='orange')), row=2, col=1)
fig.add_trace(go.Scatter(x=deu['year'], y=deu['oil_co2'],
    name='Oil', line=dict(color='brown')), row=2, col=1)

# Trade CO2
fig.add_trace(go.Scatter(x=deu['year'], y=deu['trade_co2'],
    name='Trade CO2', line=dict(color='purple'),
    showlegend=False), row=2, col=2)
fig.add_hline(y=0, line_dash='dot', line_color='gray', row=2, col=2)

fig.update_layout(
    height=700,
    title_text='Germany - Honest Decoupler?<br>'
               '<sup>Both territorial and consumption CO2 dropped - '
               'but trade dependency is growing</sup>'
)
fig.show()

# Key finding
deu_1990 = deu[deu['year'] == 1990].iloc[0]
deu_2022 = deu[deu['year'] == 2022].iloc[0]
print(f"Germany 1990-2022:")
print(f"  GDP grew:              {deu_2022['gdp_index']:.0f} (index, 1990 = 100)")
print(f"  Territorial CO2 fell:  {deu_2022['co2_index']:.0f} (index, 1990 = 100)")
print(f"  Consumption CO2 fell:  {deu_2022['consumption_co2_index']:.0f} (index, 1990 = 100)")
print(f"  Trade CO2 2022:        {deu_2022['trade_co2']:.0f} Mt (importing)")
print(f"\n Germany's decoupling is mostly real, but some emissions are being outsourced and still are")

Germany 1990-2022:
  GDP grew:              194 (index, 1990 = 100)
  Territorial CO2 fell:  63 (index, 1990 = 100)
  Consumption CO2 fell:  70 (index, 1990 = 100)
  Trade CO2 2022:        171 Mt (importing)

 Germany's decoupling is mostly real, but some emissions are being outsourced and still are


In [ ]:
# Poland - the misunderstood emitter
pol = df_indexed[df_indexed['iso_code'] == 'POL'].copy()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Three trajectories (1990 = 100)',
        'Trade CO2 - Poland exports emissions',
        'Energy mix - still coal-heavy',
        'Consumption vs Territorial gap over time'
    ]
)

# Three trajectories
fig.add_trace(go.Scatter(x=pol['year'], y=pol['gdp_index'],
    name='GDP', line=dict(color='blue', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=pol['year'], y=pol['co2_index'],
    name='Territorial CO2', line=dict(color='red', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=pol['year'], y=pol['consumption_co2_index'],
    name='Consumption CO2', line=dict(color='orange', width=2, dash='dash')), row=1, col=1)
fig.add_hline(y=100, line_dash='dot', line_color='gray', row=1, col=1)

# Trade CO2 bars
colors = ['green' if x < 0 else 'red' for x in pol['trade_co2'].fillna(0)]
fig.add_trace(go.Bar(
    x=pol['year'], y=pol['trade_co2'],
    name='Trade CO2',
    marker_color=colors,
    showlegend=False
), row=1, col=2)
fig.add_hline(y=0, line_dash='dot', line_color='black', row=1, col=2)

# Energy mix
fig.add_trace(go.Scatter(x=pol['year'], y=pol['coal_co2'],
    name='Coal', line=dict(color='black')), row=2, col=1)
fig.add_trace(go.Scatter(x=pol['year'], y=pol['gas_co2'],
    name='Gas', line=dict(color='orange')), row=2, col=1)
fig.add_trace(go.Scatter(x=pol['year'], y=pol['oil_co2'],
    name='Oil', line=dict(color='brown')), row=2, col=1)

# Consumption vs territorial gap
pol['gap'] = pol['co2_total'] - pol['consumption_co2']
fig.add_trace(go.Scatter(
    x=pol['year'], y=pol['gap'],
    name='Gap (territorial - consumption)',
    line=dict(color='green', width=2),
    fill='tozeroy',
    fillcolor='rgba(0,255,0,0.1)',
    showlegend=False
), row=2, col=2)
fig.add_hline(y=0, line_dash='dot', line_color='black', row=2, col=2)

fig.update_layout(
    height=700,
    title_text='Poland - The Misunderstood Emitter?<br>'
               '<sup>GDP grew 4x, emissions barely moved - '
               'and Poland actually exports more emissions than it imports</sup>'
)
fig.show()

pol_2022 = pol[pol['year'] == 2022].iloc[0]
pol_1990 = pol[pol['year'] == 1990].iloc[0]
print(f"Poland 1990-2022:")
print(f"  GDP grew:              {pol_2022['gdp_index']:.0f} (index, 1990 = 100)")
print(f"  Territorial CO2:       {pol_2022['co2_index']:.0f} (index, barely moved)")
print(f"  Consumption CO2:       {pol_2022['consumption_co2_index']:.0f} (index, slightly lower than territorial)")
print(f"  Trade CO2 2022:        {pol_2022['trade_co2']:.0f} Mt")
print(f"\nPoland is Europe's manufacturing floor - its 'dirty' stats")
print(f"partly reflect goods consumed by richer neighbors")

Poland 1990-2022:
  GDP grew:              396 (index, 1990 = 100)
  Territorial CO2:       84 (index, barely moved)
  Consumption CO2:       92 (index, slightly lower than territorial)
  Trade CO2 2022:        -15 Mt

Poland is Europe's manufacturing floor - its 'dirty' stats
  partly reflect goods consumed by richer neighbors
